# DDPM

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import numpy as np
from datasets import load_dataset

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import display, HTML
from tqdm import tqdm
from dlprog import train_progress

prog = train_progress()

plt.style.use("ggplot")
plt.rcParams["animation.embed_limit"] = 30.0 * 1024 * 1024

def show_images(imgs, ncol=5, w=128):
    _, _, h_img, w_img = imgs.shape
    h = int(h_img * w / w_img)
    imgs = transforms.Resize((h, w), antialias=True)(imgs)
    img = torchvision.utils.make_grid(imgs, nrow=ncol)
    img = transforms.functional.to_pil_image(img)
    display(img)

device = torch.accelerator.current_accelerator(check_available=True)
device

## MNIST Dataset

In [ ]:
SIZE = 128
batch_size = 64

ds = load_dataset("nkirschi/oxford-flowers")

transform = transforms.Compose([
    transforms.Resize(SIZE),
    transforms.CenterCrop(SIZE),
    transforms.ToTensor(),
])

def transform_batch(examples):
    examples["image"] = [transform(img.convert("RGB")) for img in examples["image"]]
    return examples

ds.set_transform(transform_batch)

dataloader = DataLoader(
    ds["train"],
    batch_size=batch_size,
    shuffle=True,
)

batch = next(iter(dataloader))
x = batch["image"]

print(f"Number of images: {len(ds['train'])}")
show_images(x[:20])

## Noise Schedule

In [ ]:
# T = 1000
# s = 0.008
# t_ = torch.arange(T + 1)
# t = torch.arange(1, T + 1)
# t_1 = t - 1
# bar_alpha = (torch.cos((t_ / T + s) / (1 + s) * np.pi / 2) ** 2) / (np.cos(s / (1 + s) * np.pi / 2) ** 2)
# alpha = bar_alpha[t] / bar_alpha[t_1]
# beta = 1 - alpha
# tilde_beta = (1 - bar_alpha[t_1]) / (1 - bar_alpha[t]) * beta
# bar_alpha = bar_alpha[t]
# plt.plot(t, alpha, label=r"$\alpha_t$")
# plt.plot(t, bar_alpha, label=r"$\bar \alpha_t$");
# plt.plot(t, tilde_beta, label=r"$\tilde \beta_t$");
# plt.xlabel(r"$t$")
# plt.legend();

In [ ]:
T = 500
beta_1 = 1e-4
beta_T = 0.02
beta = torch.linspace(beta_1, beta_T, T + 1)
alpha = 1 - beta
bar_alpha = torch.cumprod(alpha, dim=0)
t = torch.arange(1, T + 1)
t_1 = t - 1
plt.plot(t, beta[t], label=r"$\beta_t$")
plt.plot(t, alpha[t], label=r"$\alpha_t$")
plt.plot(t, bar_alpha[t], label=r"$\bar \alpha_t$")
plt.xlabel(r"$t$")
plt.legend();

## U-Net

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, d=128, max_len=T, base=10_000):
        super().__init__()
        p = torch.arange(max_len)
        i = torch.arange(0, d, 2)
        pe = torch.zeros(max_len, d)
        pe[:, ::2] = torch.sin(p.unsqueeze(-1) / base ** (i / d))
        pe[:, 1::2] = torch.cos(p.unsqueeze(-1) / base ** (i / d))
        self.register_buffer("pe", pe)

    def forward(self, t):
        return self.pe[t]

class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel_size=7, stride=2, padding=3, d_t=128):
        super().__init__()
        self.fc_time = nn.Linear(d_t, c_in)
        self.norm = nn.GroupNorm(c_in // 4 or 1, c_in)
        self.activation = nn.SiLU()
        self.conv = nn.Conv2d(c_in, c_out, kernel_size, stride, padding)

    def forward(self, x, embed_t):
        x = x + self.fc_time(embed_t)[:, :, None, None]
        x = self.norm(x)
        x = self.activation(x)
        x = self.conv(x)
        return x

class ConvTBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel_size=7, stride=2, padding=3, d_t=128):
        super().__init__()
        self.fc_time = nn.Linear(d_t, c_in)
        self.norm = nn.GroupNorm(c_in // 4, c_in)
        self.activation = nn.SiLU()
        self.conv = nn.ConvTranspose2d(c_in, c_out, kernel_size, stride, padding, output_padding=1)

    def forward(self, x, embed_t, skip=None):
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = x + self.fc_time(embed_t)[:, :, None, None]
        x = self.norm(x)
        x = self.activation(x)
        x = self.conv(x)
        return x

class AttentionBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.mha = nn.MultiheadAttention(d, 8)
        self.norm = nn.RMSNorm(d)

    def forward(self, x):
        b, c, h, w = x.shape
        x = x.reshape(b, c, -1)
        x, _ = self.mha(x, x, x)
        x = self.norm(x)
        x = x.reshape(b, c, h, w)
        return x

class UNet(nn.Module):
    def __init__(self, d_t=128):
        super().__init__()
        self.embed = TimeEmbedding(d=d_t)
        self.conv1 = ConvBlock(3, 64) # (b, 64, 64, 64)
        self.conv2 = ConvBlock(64, 128) # (b, 128, 32, 32)
        self.conv3 = ConvBlock(128, 256) # (b, 256, 16, 16)
        self.attn = AttentionBlock(16 * 16)
        self.convT3 = ConvTBlock(256, 128) # (b, 128, 32, 32)
        self.convT2 = ConvTBlock(128 * 2, 64) # (b, 64, 64, 64)
        self.convT1 = ConvTBlock(64 * 2, 32) # (b, 32, 128, 128)
        self.out_conv = ConvBlock(32, 3, 1, 1, 0) # (b, 3, 128, 128)

    def forward(self, x, t):
        # x: (b, 3, 128, 128)
        # t: (b,)
        t_embed = self.embed(t)
        z1 = self.conv1(x, t_embed) # (b, 64, 64, 64)
        z2 = self.conv2(z1, t_embed) # (b, 128, 32, 32)
        z3 = self.conv3(z2, t_embed) # (b, 128, 16, 16)
        z = self.attn(z3) + z3  # (b, 128, 16, 16)
        z = self.convT3(z, t_embed) # (b, 128, 32, 32)
        z = self.convT2(z, t_embed, skip=z2) # (b, 64, 64, 64)
        z = self.convT1(z, t_embed, skip=z1) # (b, 32, 128, 128)
        z = self.out_conv(z, t_embed) # (b, 3, 128, 128)

        return z

テスト

In [ ]:
model = UNet().to(device)
batch = next(iter(dataloader))
x = batch["image"]
t = torch.randint(0, T, (len(x),))
x = x.to(device)
with torch.no_grad():
    z = model(x, t)
print(z.shape)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

## Inference

In [ ]:
@torch.inference_mode()
def sample(model, n_samples):
    model.eval()
    x = torch.randn(n_samples, 3, SIZE, SIZE).to(device)
    hist = [x.cpu()]

    for t in tqdm(range(T, 0, -1), desc="Sampling"):
        t -= 1
        t = torch.tensor([t] * n_samples)
        beta_t = beta[t][:, None, None, None].to(device)
        alpha_t = alpha[t][:, None, None, None].to(device)
        bar_alpha_t = bar_alpha[t][:, None, None, None].to(device)
        t = t.to(device)

        eps = model(x, t)
        mu = (1 / alpha_t.sqrt()) * (x - (1 - alpha_t)/(1 - bar_alpha_t).sqrt() * eps)
        if t[0] == 0:
            noise = 0
        else:
            noise = torch.randn_like(x)
        x = mu + beta_t.sqrt() * noise
        hist.append(x.cpu())
    return x, hist


def diffusion(model, n_samples=5, duration=5):
    _, hist = sample(model, n_samples)
    fig, ax = plt.subplots(figsize=(10, 1))
    ims = []
    for x in hist:
        x = torch.clamp(x, 0, 1)
        im = ax.imshow(torchvision.utils.make_grid(x, nrow=10).permute(1, 2, 0))
        ax.axis("off")
        ims.append([im])
    ani = animation.ArtistAnimation(fig, ims, interval=int(duration*1000/len(hist)), repeat=False)
    display(HTML(ani.to_jshtml()))
    plt.close()

テスト

In [ ]:
diffusion(model)

## Training

In [ ]:
def train(model, dataloader, optimizer, n_epochs=10):
    model.train()
    prog.start(n_iter=len(dataloader), n_epochs=n_epochs, label="loss")
    for _ in range(n_epochs):
        for batch in dataloader:
            x0 = batch["image"]
            optimizer.zero_grad()
            x0 = x0.to(device)
            eps = torch.randn(x0.size()).to(device)
            t = torch.randint(0, T, (len(x0),))
            bar_alpha_t = bar_alpha[t, None, None, None].to(device)
            xt = bar_alpha_t.sqrt() * x0 + (1 - bar_alpha_t).sqrt() * eps
            pred = model(xt, t)
            loss = F.mse_loss(pred, eps)
            loss.backward()
            optimizer.step()
            prog.update(loss.item())

In [ ]:
model = UNet().to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95))
train(model, dataloader, optimizer, n_epochs=1)

In [ ]:
diffusion(model)